# Step 09.3 — LSTM Cell State and Gates：给长期信息一条可控制的路径

Step 9.1–9.2 已经建立问题：Vanilla RNN 的长期 credit assignment 要穿过许多 recurrent Jacobians；tanh saturation、state 与 direction 都会影响 long-range gradient contribution。

这一小节只学习 LSTM（Long Short-Term Memory）为什么引入第二个 state c_t，以及 forget / input / output gates 怎样控制读写。不会进入 Step 9.4。

继续沿用记号：B=batch size，T=sequence length，t=0...T-1，C=input feature width，H=hidden width；h_-1 与 c_-1 是初始 states，h_(T-1) 与 c_(T-1) 是末态。

这一讲按自然顺序组织为：动机 → gate 基础 → equations/shape → 极小数值更新 → direct cell-state gradient path → MLX 0.32.2 映射。


## 1. 为什么 LSTM 要多一个 Cell State c_t？

Vanilla RNN 每一步都把旧历史与新输入混合后整体经过 tanh：

$$
h_t = \tanh(W_x x_t + W_h h_{t-1} + b).
$$

Step 9.2 已经看到，长期 gradient contribution 必须穿过许多 recurrent transformations 与 activation derivatives。

LSTM 把 recurrent state 扩展成两个：

- h_t：hidden state，shape 为 (B,H)。
- c_t：cell state，shape 也为 (B,H)。

c_t 提供更直接的内部 state-update 路径；h_t 是从当前 c_t 经过 output gate 过滤后的 hidden output，并继续参与下一时间步的 gate 计算。

可以先把结构理解为：c_(t-1) 经过 retain + write 得到 c_t，再由 c_t 产生过滤后的 h_t。

但不要把 c_t 和 h_t 机械等同为“长期记忆”和“短期记忆”；这是入门比喻，不是严格数学分类。


## 2. Gate 在数学上是什么？先解释 Sigmoid 与逐元素乘法

LSTM 的三个真正 gates 使用 sigmoid：

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

它把任意实数映射到 (0,1)。例如 σ(-2)≈0.1192，σ(0)=0.5，σ(2)≈0.8808。

因此 gate 是 soft coefficient（软控制系数），不是只能 0/1 的 binary switch。一个 gate 通常是 (B,H)，每个 sample、每个 hidden coordinate 都可以有自己的控制值。

符号 ⊙ 表示 Hadamard product / element-wise multiplication。比如 [0.9,0.2]^T ⊙ [10,-5]^T = [9,-1]^T；它不是 matrix multiplication。

LSTM 四个分量为：

$$
i_t=\sigma(W_{xi}x_t+W_{hi}h_{t-1}+b_i)
$$
$$
f_t=\sigma(W_{xf}x_t+W_{hf}h_{t-1}+b_f)
$$
$$
g_t=\tanh(W_{xg}x_t+W_{hg}h_{t-1}+b_g)
$$
$$
o_t=\sigma(W_{xo}x_t+W_{ho}h_{t-1}+b_o)
$$

其中 f_t=forget gate，i_t=input gate，o_t=output gate；g_t 是 tanh candidate，不是 sigmoid gate。

核心更新：

$$
\boxed{c_t=f_t\odot c_{t-1}+i_t\odot g_t}
$$
$$
\boxed{h_t=o_t\odot\tanh(c_t)}
$$


In [ ]:
import math

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

for z in [-2.0, 0.0, 2.0]:
    print(f'z={z:>4.1f} -> sigmoid(z)={sigmoid(z):.6f}')


## 3. Shape 与 Packed 4H Projection，再做一个极小数值更新

列向量记号下，x_t=(C,1)，h_(t-1)=(H,1)，c_(t-1)=(H,1)。每一个 gate/candidate 都要产生 H 个值，所以例如 W_xi=(H,C)、W_hi=(H,H)、b_i=(H,1)。

工程实现通常把四组 input projections 打包成 W_x=(4H,C)，把四组 recurrent projections 打包成 W_h=(4H,H)。

在 MLX row/batch convention 中：x_t=(B,C)，h_prev=(B,H)，两条 projection 相加后得到 (B,4H)，再 split 成 i/f/g/o 四个 (B,H) tensors。

这里的 4H 不是新的 hidden width，只是临时把四组 H-dimensional preactivations 放在一起。

极小 H=1 数值例子。假设 gate/candidate outputs 已经是：c_(t-1)=0.80，f_t=0.90，i_t=0.20，g_t=0.50，o_t=0.70。

旧 state 保留量：0.9×0.8=0.72；新写入量：0.2×0.5=0.10。

因此：

$$
c_t=0.72+0.10=0.82.
$$

以及：

$$
h_t=0.7\tanh(0.82)\approx0.4725.
$$

这就是 retain（保留）+ write（写入）+ expose（输出）的最小直觉。


In [ ]:
import mlx.core as mx

c_prev = mx.array([0.80])
f_t = mx.array([0.90])
i_t = mx.array([0.20])
g_t = mx.array([0.50])
o_t = mx.array([0.70])

retained = f_t * c_prev
written = i_t * g_t
c_t = retained + written
h_t = o_t * mx.tanh(c_t)

print('retained old state:', retained)
print('written candidate :', written)
print('new cell state    :', c_t)
print('new hidden state  :', h_t)


## 4. 为什么 Cell State 对长期 Gradient 更友好？只分析 Direct Path

从 c_t=f_t⊙c_(t-1)+i_t⊙g_t 出发，只看计算图中 c_(t-1) 直接乘 f_t 到 c_t 的那条 edge。

在这个局部 partial derivative 中，把 h_(t-1) 与当前 gate values 当作给定的其他输入，则 direct cell-state Jacobian contribution 为：

$$
\boxed{\left.\frac{\partial c_t}{\partial c_{t-1}}\right|_{direct}=\operatorname{diag}(f_t)}
$$

所以 direct backward contribution 为：

$$
g_{c_{t-1}}^{direct}=\operatorname{diag}(f_t)g_{c_t}=f_t\odot g_{c_t}.
$$

若某个 coordinate 的 f_t≈1，这条 direct path 可以接近 identity，使旧 cell value 与相应 gradient contribution 更容易继续传递；若 f_t≈0，则模型主动忘掉对应旧内容。

例如 f_t=[0.95,0.20]^T，upstream cell gradient g_c_t=[2,-3]^T，则 direct contribution 是 [1.90,-0.60]^T。

这就是 LSTM 相比 Vanilla RNN 的重要设计思想：把“保留多少旧状态”变成显式、可学习的控制。

但必须保留严谨边界：diag(f_t) 只是 direct cell-state edge contribution，不是完整 LSTM total Jacobian；gates 自己依赖 x_t 与 h_(t-1)，完整 BPTT 还有其他路径。连续多个 f_t<1 仍可能衰减，LSTM 也不保证彻底消除 vanishing/exploding，更不是无限无损 memory。


In [ ]:
f_t = mx.array([0.95, 0.20])
g_c_t = mx.array([2.0, -3.0])

g_c_prev_direct = f_t * g_c_t

print('forget gate:', f_t)
print('upstream cell gradient:', g_c_t)
print('direct gradient contribution:', g_c_prev_direct)


## 5. 用 Apple MLX 写出 Gates，并对照 MLX 0.32.2 的 nn.LSTM

为了看清 gates，先写一个教学版 EducationalLSTMCell。它使用两个 packed Linear projections：x_proj: C→4H，h_proj: H→4H。

mx.split(packed, 4, axis=-1) 的用途是沿最后一个 axis 把 (B,4H) 平均拆成 4 个 (B,H) tensors。

MLX nn.Linear(input_dims, output_dims) 的 weight 存储 shape 是 (output_dims,input_dims)，所以这里 x_proj.weight=(4H,C)，h_proj.weight=(4H,H)。


In [ ]:
import mlx.nn as nn

class EducationalLSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.x_proj = nn.Linear(input_size, 4 * hidden_size, bias=True)
        self.h_proj = nn.Linear(hidden_size, 4 * hidden_size, bias=False)

    def __call__(self, x_t, h_prev, c_prev):
        packed = self.x_proj(x_t) + self.h_proj(h_prev)
        i_pre, f_pre, g_pre, o_pre = mx.split(packed, 4, axis=-1)

        i_t = mx.sigmoid(i_pre)
        f_t = mx.sigmoid(f_pre)
        g_t = mx.tanh(g_pre)
        o_t = mx.sigmoid(o_pre)

        c_t = f_t * c_prev + i_t * g_t
        h_t = o_t * mx.tanh(c_t)
        return h_t, c_t, (i_t, f_t, g_t, o_t)

B, C, H = 2, 3, 4
cell = EducationalLSTMCell(C, H)
x_t = mx.zeros((B, C))
h_prev = mx.zeros((B, H))
c_prev = mx.zeros((B, H))
h_t, c_t, gates = cell(x_t, h_prev, c_prev)

print('x_t:', x_t.shape)
print('h_prev / c_prev:', h_prev.shape, c_prev.shape)
print('h_t / c_t:', h_t.shape, c_t.shape)
print('gate shapes:', [g.shape for g in gates])


项目固定 mlx==0.32.2。该版本内置 API 是 nn.LSTM(input_size, hidden_size, bias=True)。

batched input 可以是 (B,T,C)；可选 initial hidden 与 cell 为 (B,H)。返回 all_hidden 与 all_cell，二者都是 (B,T,H)。

MLX 0.32.2 源码把参数打包为 Wx=(4H,C)、Wh=(4H,H)、bias=(4H,)（bias=True 时），并按 i,f,g,o 顺序 split。

如果 initial hidden/cell 不提供，第一步等价于没有上一 hidden contribution，以及 zero previous cell contribution。

下面只给静态 API 形状示例；母版不执行。


In [ ]:
B, T, C, H = 2, 5, 3, 4

lstm = nn.LSTM(input_size=C, hidden_size=H, bias=True)
X = mx.zeros((B, T, C))
h_minus_1 = mx.zeros((B, H))
c_minus_1 = mx.zeros((B, H))

all_hidden, all_cell = lstm(
    X,
    hidden=h_minus_1,
    cell=c_minus_1,
)

print('X:', X.shape)
print('Wx / Wh / bias:', lstm.Wx.shape, lstm.Wh.shape, lstm.bias.shape)
print('all_hidden:', all_hidden.shape)
print('all_cell:', all_cell.shape)


### Step 9.3 材料边界

- LSTM 在 h_t 之外引入 c_t；两者通常都是 (B,H)。
- sigmoid gate 是 (0,1) 的 soft coefficient；⊙ 是 element-wise multiplication。
- forget gate 控制旧 cell 的直接保留，input gate 控制 candidate 写入，output gate 控制 c_t 暴露到 h_t 的程度。
- candidate g_t 使用 tanh，不是 sigmoid gate。
- direct cell-state edge 的 gradient contribution 由 f_t 逐元素控制；它提供更直接的可学习保留路径，但不是完整 LSTM Jacobian，也不保证彻底消除长期梯度问题。
- MLX 0.32.2 nn.LSTM 使用 packed (4H,C)/(4H,H) parameters，并返回 hidden-state sequence 与 cell-state sequence。
- 本轮只准备 Step 9.3；不进入 Step 9.4，也不据此声明学员已经掌握或完成练习。
